In [0]:
%pip install ruamel.yaml openapi-spec-validator strands-agents python-dotenv openai -q

In [0]:
%restart_python

In [0]:
import sys, os, importlib

# Add the Standard-Agent directory to sys.path
project_dir = "/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent"
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

# Get auth from Databricks SDK (works on serverless, clusters, and apps)
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
auth = w.config.authenticate()

# SDK >=0.67 returns a dict; older versions return a callable
if callable(auth):
    headers = {}
    auth(headers)
else:
    headers = auth

token = headers.get("Authorization", "").removeprefix("Bearer ")
host = w.config.host

# Set env vars that the portable agent reads
os.environ["OPENAI_API_KEY"] = token
os.environ["OPENAI_BASE_URL"] = f"{host}/serving-endpoints"
os.environ["MODEL_ID"] = "databricks-claude-sonnet-4"

print(f"Host: {host}")
print(f"Token: {'set' if token else 'NOT SET'} ({len(token)} chars)")
print(f"Model: {os.environ['MODEL_ID']}")
print()

# Force-reload ALL agent modules to pick up changes
import agent.tools
importlib.reload(agent.tools)
import agent.system_prompt
importlib.reload(agent.system_prompt)
import agent.agent
importlib.reload(agent.agent)

from agent.agent import create_agent

iri_agent = create_agent()
print("\n\u2705 Agent ready for testing")

In [0]:
# Test: GitHub URL-based spec review
# The agent should detect the URL, use fetch_yaml_from_url to download it,
# then proceed with the standard review workflow.

github_url = "https://github.com/Insured-Retirement-Institute/One-Time-Fund-Transfer/blob/edwardmruiz-2/FundTransfer_v1.1.0.yml"

question = f"Review this spec for style guide compliance: {github_url}\nThis is a revision — no previous findings to compare. Yes, check cross-spec consistency."

result = iri_agent(question)
print(str(result))